# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hashir9099/flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one piece of content (content_id), belonging to one client (client_id). Metrics are aggregated over a rolling 90-day window (_90d columns), with a _last_30d vs _prev_30d split for trend comparison. There's no absolute calendar date — the window is relative, not anchored.

In [18]:
import os

if not os.path.exists("flyrank-ml"):
    !git clone https://github.com/Hashir9099/flyrank-ml.git

os.chdir("flyrank-ml")
print(os.getcwd())

Cloning into 'flyrank-ml'...
remote: Enumerating objects: 191, done.
remote: Counting objects: 100% (191/191), done.
remote: Compressing objects: 100% (141/141), done.
remote: Total 191 (delta 80), reused 100 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (191/191), 1.90 MiB | 5.22 MiB/s, done.
Resolving deltas: 100% (80/80), done.
/content/flyrank-ml/flyrank-ml/flyrank-ml


In [19]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))
print(df.columns.tolist())
print(df.head())

Rows: 30000
Columns: 44
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
             content_id          client_id  search_volume  competition  \
0  content_304f48230142  client_f369cb89fc           10.0         0.67   
1  content_a1fb4e703a9e  client_4e07408562           90.0

In [20]:
print("Rows:", len(df))
print("Unique content_id:", df["content_id"].nunique())
print("Unique client_id:", df["client_id"].nunique())
print("Pages per client (describe):")
print(df.groupby("client_id").size().describe())

Rows: 30000
Unique content_id: 30000
Unique client_id: 32
Pages per client (describe):
count      32.000000
mean      937.500000
std      1376.387113
min         3.000000
25%       110.250000
50%       567.000000
75%      1058.750000
max      7008.000000
dtype: float64


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Label:** `trend_direction` — the pipeline defines `is_declining_label = (trend_direction == "down")`.

**Excluded (leakage):** `trend_pct` — this is the number `trend_direction` was derived from, so
including it would let the model see the label in disguise.

**Context (identifiers, not predictive):** `content_id`, `client_id`, `provider_used`, `model_used`
— these identify the row/generation source but shouldn't drive a prediction.

**Feature (candidates):** search_volume, competition, competition_level, cpc, content_type,
main_intent, word_count, char_count, impressions_90d, clicks_90d, pageviews_90d, sessions_90d,
users_90d, engaged_sessions_90d, ai_sessions_90d, scroll_events_90d, days_with_impressions,
days_with_sessions, impressions_last_30d, clicks_last_30d, sessions_last_30d,
impressions_prev_30d, clicks_prev_30d, sessions_prev_30d, content_age_days, age_tier,
age_tier_order, days_since_last_update, freshness_tier, word_count_tier, char_count_tier,
ctr, avg_position, engagement_rate, scroll_rate, ai_traffic_pct, impression_tier, position_tier

**Note:** `ctr` and `avg_position` are borderline — they may be computed over the same window
`trend_direction` trends over, so they could carry partial signal about the label rather than
being fully independent features. Worth flagging as a soft leakage risk, not a hard exclusion.

In [21]:
label_col = "trend_direction"
excluded = ["trend_pct"]
context = ["content_id", "client_id", "provider_used", "model_used"]

feature = [c for c in df.columns if c not in [label_col] + excluded + context]

print("Label:", label_col)
print("Excluded:", excluded)
print("Context:", context)
print("Feature count:", len(feature))
print(feature)

all_buckets = [label_col] + excluded + context + feature
print("All 44 columns accounted for ✔️" if sorted(all_buckets) == sorted(df.columns) else "Mismatch — check for a missing/duplicated column")

Label: trend_direction
Excluded: ['trend_pct']
Context: ['content_id', 'client_id', 'provider_used', 'model_used']
Feature count: 38
['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier']
All 44 columns accounted for ✔️


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Every claim from Sections 1 and 2 gets checked here against the actual data:

- **Grain:** confirmed one row per `content_id` (no duplicates).
- **Missing values:** `word_count` and `char_count` (and their tier columns) have nulls —
  noted below with exact counts.
- **Label balance:** `trend_direction` distribution shown below — not a huge class imbalance,
  but worth knowing before modeling.
- **Label consistency:** `trend_pct` sign/magnitude lines up with `trend_direction` category,
  confirming the label was derived correctly and consistently.

In [22]:
# grain check
print("Duplicate content_id rows:", df.duplicated(subset=["content_id"]).sum())

# missing values
nulls = df.isna().sum()
print("Columns with missing values:")
print(nulls[nulls > 0])

# label balance
print("\ntrend_direction counts:")
print(df["trend_direction"].value_counts())
print(df["trend_direction"].value_counts(normalize=True).round(3) * 100)

# label consistency check
print("\ntrend_pct by trend_direction:")
print(df.groupby("trend_direction")["trend_pct"].describe())

Duplicate content_id rows: 0
Columns with missing values:
search_volume         2468
competition           2468
competition_level     2610
cpc                   2468
main_intent           2374
word_count            7699
char_count            7699
provider_used        21438
model_used            5733
word_count_tier       7699
char_count_tier       7699
scroll_rate            125
trend_pct             3388
dtype: int64

trend_direction counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64
trend_direction
down      54.2
stable    19.9
up        14.6
new        7.5
flat       3.8
Name: proportion, dtype: float64

trend_pct by trend_direction:
                   count        mean          std    min   25%    50%    75%  \
trend_direction                                                                
down             16262.0  -58.113830    23.488605 -100.0 -75.9 -55.60  -38.5   
flat                 0.0         NaN

What this dataset can't tell you:

- **No absolute time window:** every time-based column is relative (`_90d`, `_last_30d`,
  `_prev_30d`) — there's no calendar date, so you can't say "as of [date]" or compare this
  snapshot to a different time period.
- **Uneven pages per client:** client history isn't balanced — some clients have far more
  pages represented than others (see distribution below), so patterns could be dominated by
  a handful of large clients rather than reflecting the average client.
- **Provider/model concentration:** `provider_used` / `model_used` may be concentrated in a
  few values — if so, findings may reflect one tool's behavior more than "content quality"
  broadly.
- **Missing values are structural, not random:** the nulls in `word_count`/`char_count`
  (Section 3) may correlate with `content_type` or `provider_used` rather than being random —
  worth treating as a signal, not just something to drop.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [23]:
print("Pages per client:")
print(df.groupby("client_id").size().describe())

print("\nprovider_used:")
print(df["provider_used"].value_counts(normalize=True).round(3) * 100)

print("\nmodel_used:")
print(df["model_used"].value_counts(normalize=True).round(3) * 100)

print("\nnull word_count by content_type:")
print(df[df["word_count"].isna()]["content_type"].value_counts())

Pages per client:
count      32.000000
mean      937.500000
std      1376.387113
min         3.000000
25%       110.250000
50%       567.000000
75%      1058.750000
max      7008.000000
dtype: float64

provider_used:
provider_used
google    86.0
openai    14.0
Name: proportion, dtype: float64

model_used:
model_used
gemini-3-flash-preview    54.7
gpt-4o-mini               20.5
gemini-2.5-flash          15.1
gpt-5-mini                 6.6
unknown                    3.1
Name: proportion, dtype: float64

null word_count by content_type:
content_type
keyword article    7699
Name: count, dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.